# Mooring Field Detection — Kaggle GPU Pipeline

**Before running:**
1. Settings → Accelerator → **GPU T4** (or P100)
2. Settings → Internet → **On**
3. **Add data** → attach your `mooring-field-data` dataset (`data/imagery` + `data/labels`)
4. Add-ons → Secrets → `GOOGLE_MAPS_API_KEY` only if you will run evaluate/fetch

Run cells top to bottom. Do **not** restart the kernel mid-session.

This notebook calls package entry points in `src/mooring_fields/` (see `docs/KAGGLE.md`).

In [ ]:
# Cell 1 — Clone repo and install (must run before Cell 2)
import subprocess, sys, shutil, os
from pathlib import Path

REPO = "/kaggle/working/MooringFieldDetection"
GITHUB_URL = "https://github.com/IshanKasam/MooringFieldDetection.git"

if Path(REPO).exists():
    shutil.rmtree(REPO)

subprocess.run(["git", "clone", GITHUB_URL, REPO], check=True)
os.chdir(REPO)
# Make imports work even if editable install is flaky on Kaggle
src = str(Path(REPO) / "src")
if src not in sys.path:
    sys.path.insert(0, src)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "python-dotenv"], check=True)

import mooring_fields
print("install done; mooring_fields at", mooring_fields.__file__)
print("cwd:", os.getcwd())

In [ ]:
# Cell 2 — Bootstrap: link Kaggle dataset, detect GPU
# Implemented in: src/mooring_fields/runtime.py → bootstrap_kaggle()
import sys, json, os
from pathlib import Path

REPO = "/kaggle/working/MooringFieldDetection"
os.chdir(REPO)
src = str(Path(REPO) / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from mooring_fields.runtime import bootstrap_kaggle

print(json.dumps(bootstrap_kaggle(), indent=2))
# Expect: cuda true, has_imagery true, device 0

In [ ]:
# Cell 3 — Train on corrected labels (data/labels)
# Implemented in: src/mooring_fields/train_boats.py → train()
import sys, json, os
from pathlib import Path

REPO = "/kaggle/working/MooringFieldDetection"
os.chdir(REPO)
src = str(Path(REPO) / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from mooring_fields.train_boats import train
from mooring_fields.runtime import publish_outputs

report = train(use_corrected_labels=True)
report["published"] = publish_outputs()
print(json.dumps({k: v for k, v in report.items() if k != "results"}, indent=2))

In [ ]:
# Cell 4 — Evaluate (optional; needs GOOGLE_MAPS_API_KEY secret)
# Implemented in: src/mooring_fields/evaluate.py → evaluate_val()
import sys, json, os
from pathlib import Path

REPO = "/kaggle/working/MooringFieldDetection"
os.chdir(REPO)
src = str(Path(REPO) / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from mooring_fields.evaluate import evaluate_val
from mooring_fields.runtime import publish_outputs

report = evaluate_val()
report["published"] = publish_outputs()
summary = {k: v for k, v in report.items() if k not in ("per_site", "clusters")}
print(json.dumps(summary, indent=2))
# Outputs under /kaggle/working/mooring_outputs/

## Download outputs

After **Save Version**, open the version → **Output** → `mooring_outputs/`:

- `mooring_boats/weights/best.pt`
- `evaluation_results.json` / `evaluation_clusters.kml` (if you ran evaluate)